# ESCOPE: detección de anomalías por tamaño de clúster

Compara **Mahalanobis con covarianza Ledoit–Wolf**, **Isolation Forest** y **Local Outlier Factor (LOF, modo de novedad)** sobre clústeres BIRCH de Euclid Q1. Reutiliza las asignaciones de ESCOPE: catálogo filtrado v3.2, PCA 40D estandarizadas, umbral 5.5, ramificación 42, lote 25.000. No necesita ESCOPE abierto ni recalcular BIRCH. [Detectores](https://scikit-learn.org/stable/modules/outlier_detection.html) · [Covarianza regularizada](https://scikit-learn.org/stable/modules/generated/sklearn.covariance.LedoitWolf.html)

- **Pequeños:** 50–149 objetos. **Grandes:** 500 o más. Sin seleccionar por grado, enriquecimiento ni score.
- El **clúster 380** se informa aparte como caso ilustrativo, no entra en la comparación por tamaño.
- Ajuste no supervisado por clúster, con dos mitades complementarias y tres repeticiones. Cada objeto se evalúa en una mitad distinta de la que ajustó ese detector. Se inspecciona el 10 % superior de cada mitad, redondeado hacia arriba.
- **Mitad completa:** se aprovecha toda la mitad de ajuste. **25 por ajuste:** sensibilidad con el mismo tamaño de ajuste en todos los grupos, conservando exactamente las mismas mitades de evaluación. El subsample se toma sin mirar etiquetas.
- Se mide **recuperación de referencias A+B y A/B/C**, no precisión, tasa de falsos positivos ni capacidad demostrada de descubrir lentes. Unknown no es una etiqueta negativa. El PCA, la agrupación y el filtro v3.2 ya existen antes del experimento; el filtro de producción protege referencias conocidas. Por tanto, este análisis es retrospectivo, no una prueba ciega externa ni causal sobre el tamaño.

## 1. Archivos y configuración

No instala ni actualiza librerías. Usa Python 3.11+, NumPy, pandas, PyArrow y scikit-learn ya presentes. En Colab, monta Drive y ajusta las tres rutas si lo necesitas. Los valores por defecto buscan el PCA y el CSV en sus carpetas de catálogo habituales, y el NPZ en `MyDrive/catalogues/morphology_catalogue`. También puedes colocar los tres archivos en un mismo directorio y definir `ESCOPE_ANOMALY_DATA` con esa ruta.

El NPZ es la caché BIRCH de ESCOPE (no es un modelo serializado con pickle). El nombre original es `27671cac08a351324a6c813a.npz`. Se comprueban SHA-256 de los tres archivos y el hash ordenado de `id_str` antes de unir los objetos con sus clústeres. Si falla la verificación, no se reasignan clústeres por fila ni se continúa con otros datos.

Para repetir desde el bucket con acceso autorizado se puede usar `gcloud storage cp` con las tres rutas de `GCS_SOURCE`; el registro de ejecución incluye sus hashes y las versiones de librerías. Los CSV se escriben en `ESCOPE_ANOMALY_OUTPUT` o en el subdirectorio indicado en la configuración.

In [ ]:
import hashlib
import json
import math
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from threadpoolctl import threadpool_limits

ROOT = Path(os.environ.get("ESCOPE_ANOMALY_DATA", "/content/drive/MyDrive/catalogues"))
PCA_NAME = "representations_pca_40_artifacts_filtered_v3_2_continuous_multiscale_hough_lines.parquet"
REF_NAME = "q1_discovery_engine_lens_catalog.csv"
CACHE_NAME = "27671cac08a351324a6c813a.npz"
PCA_PATH = ROOT / PCA_NAME if (ROOT / PCA_NAME).is_file() else ROOT / "morphology_catalogue" / PCA_NAME
REFERENCE_PATH = ROOT / REF_NAME if (ROOT / REF_NAME).is_file() else ROOT / "strong_lensing_catalogue" / REF_NAME
ASSIGNMENT_PATH = ROOT / CACHE_NAME if (ROOT / CACHE_NAME).is_file() else ROOT / "morphology_catalogue" / CACHE_NAME
OUTPUT = Path(os.environ.get("ESCOPE_ANOMALY_OUTPUT", str(ROOT / "escope_anomaly_by_cluster_size")))

if not ROOT.exists() and str(ROOT).startswith("/content/drive/"):
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        drive.mount("/content/drive")
        if not PCA_PATH.exists() and (ROOT / PCA_NAME).exists():
            PCA_PATH = ROOT / PCA_NAME
        if not REFERENCE_PATH.exists() and (ROOT / REF_NAME).exists():
            REFERENCE_PATH = ROOT / REF_NAME
        if not ASSIGNMENT_PATH.exists() and (ROOT / CACHE_NAME).exists():
            ASSIGNMENT_PATH = ROOT / CACHE_NAME

BUCKET = "gs://euclid-umap-ivan-0424-data"
GCS_SOURCE = [
    f"{BUCKET}/catalogues/morphology_catalogue/{PCA_NAME}",
    f"{BUCKET}/catalogues/strong_lensing_catalogue/{REF_NAME}",
    f"{BUCKET}/analysis_cache/birch/v2/{CACHE_NAME}",
]
INPUTS = {
    "pca": (PCA_PATH, "61f2258374a9e3c830c1cab724e1eb597d00d95e29889e051ec67d83558178aa"),
    "reference": (REFERENCE_PATH, "ee5e60cd507413eabf3da8ffa37a3c527212da00feb29083cf516a86d6f26877"),
    "birch": (ASSIGNMENT_PATH, "7e73011c3abdb3cd01250c1f0f894efc07ca3cf4ddf11953ce462576fc7eb8e7"),
}

SEEDS = (20261125, 20261126, 20261127)
FEATURES = [f"feat_pca_{i}" for i in range(40)]
SMALL_MIN, SMALL_MAX, LARGE_MIN = 50, 149, 500
CASE_CLUSTER = 380
BUDGET = 0.10
TRAIN_SIZES = {"natural": None, "matched25": 25}
METHODS = ("Mahalanobis (LW)", "Isolation Forest", "LOF")
LOF_NEIGHBORS, IF_TREES = 10, 64
COHORTS = ("small_50_149", "large_500_plus", "case_380")

## 2. Validar y cruzar los datos

El hash de secuencia usa el mismo algoritmo que la caché BIRCH. Los IDs de objetos se leen como cadenas (nunca como números en coma flotante). Se conserva el grado del CSV de referencia. El escalado original se reutiliza y no se reajusta por clúster.

In [ ]:
def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def ordered_id_digest(values):
    digest = hashlib.sha256()
    for value in values:
        data = str(value).encode("utf-8")
        digest.update(len(data).to_bytes(4, "little"))
        digest.update(data)
    return digest.hexdigest()


def load_inputs():
    hashes = {}
    for key, (path, expected_hash) in INPUTS.items():
        if not path.is_file():
            raise FileNotFoundError(f"Falta {key}: {path}. Revisa la configuración de la primera celda.")
        hashes[key] = sha256_file(path)
        if hashes[key] != expected_hash:
            raise ValueError(f"SHA-256 distinto del snapshot esperado: {path}")

    with np.load(ASSIGNMENT_PATH, allow_pickle=False) as cache:
        manifest = json.loads(cache["manifest"].tobytes())
        labels, mean, scale = (cache[key] for key in ("labels", "scaler_mean", "scaler_scale"))
    expected = {
        "schema": 2, "threshold": 5.5, "branching_factor": 42, "batch_size": 25000,
        "scaling": "standard", "features": FEATURES, "n_rows": 374982,
    }
    if any(manifest.get(key) != value for key, value in expected.items()):
        raise ValueError("La caché no es la de BIRCH 5.5 / 42 / 25000, v3.2, 40 PCA.")
    if mean.shape != (40,) or scale.shape != (40,) or not np.isfinite(mean).all() or not np.all(scale > 0):
        raise ValueError("Escalado de la caché incompleto.")

    frame = pd.read_parquet(PCA_PATH, columns=["id_str", "object_id", *FEATURES])
    frame = frame.dropna(subset=FEATURES).reset_index(drop=True)
    if labels.shape != (len(frame),) or ordered_id_digest(frame["id_str"]) != manifest["ordered_id_sha256"]:
        raise ValueError("Las filas PCA no coinciden en número y orden con la caché BIRCH.")
    object_ids = frame["object_id"].astype("string").str.strip()
    suffix = frame["id_str"].astype("string").str.rsplit("_", n=1).str[-1]
    if (not object_ids.str.fullmatch(r"[+-]?[0-9]+").fillna(False).all()
            or not object_ids.eq(suffix).fillna(False).all() or object_ids.duplicated().any()):
        raise ValueError("ID PCA inválido, distinto de su id_str o repetido.")

    refs = pd.read_csv(REFERENCE_PATH, dtype="string", usecols=["object_id", "grade"])
    refs["object_id"] = refs["object_id"].str.strip()
    refs["grade"] = refs["grade"].str.strip().str.upper()
    refs = refs.loc[refs["grade"].isin(["A", "B", "C"]) & refs["object_id"].isin(object_ids)]
    if refs.groupby("object_id")["grade"].nunique().gt(1).any():
        raise ValueError("Grados contradictorios en los objetos cruzados.")
    grades = object_ids.map(refs.drop_duplicates("object_id").set_index("object_id")["grade"]).fillna("Unknown")
    data = pd.DataFrame({"object_id": object_ids, "cluster": labels, "grade": grades})
    if data["cluster"].nunique() != 2174 or grades.value_counts().to_dict() != {
        "A": 156, "B": 163, "C": 1097, "Unknown": 373566,
    }:
        raise ValueError("El cruce no coincide con los recuentos esperados de ESCOPE.")
    raw = frame[FEATURES].to_numpy(dtype=np.float32, copy=True)
    raw -= mean
    raw /= scale
    if not np.isfinite(raw).all():
        raise ValueError("El catálogo contiene componentes PCA no finitas.")
    return data, raw, manifest, hashes


def cohort(cluster, n_objects):
    if SMALL_MIN <= n_objects <= SMALL_MAX:
        return COHORTS[0]
    if n_objects >= LARGE_MIN:
        return COHORTS[1]
    if cluster == CASE_CLUSTER:
        return COHORTS[2]
    return "not_compared"


data, features, manifest, hashes = load_inputs()
inventory = (
    data.assign(is_a=data.grade.eq("A"), is_b=data.grade.eq("B"), is_c=data.grade.eq("C"))
    .groupby("cluster").agg(n_objects=("object_id", "size"), A=("is_a", "sum"),
                            B=("is_b", "sum"), C=("is_c", "sum")).reset_index()
)
inventory["cohort"] = [cohort(c, n) for c, n in zip(inventory.cluster, inventory.n_objects)]
print(inventory.groupby("cohort").agg(clusters=("cluster", "size"), objects=("n_objects", "sum"),
                                      A=("A", "sum"), B=("B", "sum"), C=("C", "sum")).to_string())

## 3. Ajustar y evaluar por clúster

Los tres detectores ven exactamente los mismos objetos de ajuste y de evaluación en cada partición. En caso de empate en score se conserva el orden aleatorio de esa mitad. Para cada presupuesto se calcula además la recuperación esperada si se escogiesen objetos al azar **en las mismas mitades**. Los candidatos de referencia pueden estar en una mitad de ajuste, pero ese ajuste desconoce su grado.

Las repeticiones comparten objetos: muestran estabilidad frente a las particiones, **no** son observaciones independientes. El agregado es *micro*: suma referencias y aciertos antes de dividir. Un clúster sin referencias consume presupuesto, aunque no aporta al denominador de recuperación.

In [ ]:
def anomaly_scores(train, evaluate, seed):
    covariance = LedoitWolf().fit(train)
    forest = IsolationForest(n_estimators=IF_TREES, random_state=seed, n_jobs=1).fit(train)
    lof = LocalOutlierFactor(n_neighbors=LOF_NEIGHBORS, novelty=True, algorithm="brute", n_jobs=1).fit(train)
    return (
        covariance.mahalanobis(evaluate),
        -forest.score_samples(evaluate),
        -lof.score_samples(evaluate),
    )


def review_budget(n):
    return math.ceil(BUDGET * n)


def partitions(n, seed, cluster):
    rng = np.random.default_rng(np.random.SeedSequence((int(seed), int(cluster))))
    first, second = np.array_split(rng.permutation(n), 2)
    return ((first, second), (second, first))


def grade_counts(grades):
    return np.array([(grades == "A").sum(), (grades == "B").sum(), (grades == "C").sum()])


def compare_cluster(x, ids, grades, cluster, band):
    metrics, shortlist = [], []
    for seed in SEEDS:
        halves = partitions(len(x), seed, cluster)
        for fold, (train, evaluate) in enumerate(halves):
            n_review = review_budget(len(evaluate))
            counts = grade_counts(grades[evaluate])
            for protocol, cap in TRAIN_SIZES.items():
                fit = train if cap is None else train[:cap]
                base = {
                    "cohort": band, "cluster": cluster, "seed": seed, "fold": fold,
                    "protocol": protocol, "n_train": len(fit), "n_eval": len(evaluate),
                    "n_review": n_review,
                    "ref_ab": counts[:2].sum(), "ref_abc": counts.sum(),
                    "chance_ab": counts[:2].sum() * n_review / len(evaluate),
                    "chance_abc": counts.sum() * n_review / len(evaluate),
                }
                for method, values in zip(METHODS, anomaly_scores(x[fit], x[evaluate], seed + fold)):
                    if not np.isfinite(values).all():
                        raise ValueError(f"Score inválido: {method}, clúster {cluster}.")
                    ordered = np.argsort(-values, kind="stable")[:n_review]
                    chosen = evaluate[ordered]
                    hits = grade_counts(grades[chosen])
                    metrics.append(dict(base, method=method, hit_a=hits[0], hit_b=hits[1], hit_c=hits[2],
                                        hit_ab=hits[:2].sum(), hit_abc=hits.sum()))
                    if seed == SEEDS[0] and protocol == "natural":
                        shortlist.extend({
                            "cohort": band, "cluster": cluster, "fold": fold, "method": method,
                            "rank_in_fold": rank, "object_id": ids[idx], "score": float(values[local]),
                        } for rank, (idx, local) in enumerate(zip(chosen, ordered), start=1) if grades[idx] == "Unknown")
    return metrics, shortlist


groups = data.groupby("cluster", sort=True).indices
eligible = list(inventory.loc[inventory.cohort.isin(COHORTS), ["cluster", "cohort"]].itertuples(index=False, name=None))
all_ids = data.object_id.to_numpy(dtype=str)
all_grades = data.grade.to_numpy(dtype=str)
rows, candidates = [], []
with threadpool_limits(limits=1):
    for position, (cluster, band) in enumerate(eligible, 1):
        ix = groups[cluster]
        metrics, shortlist = compare_cluster(features[ix], all_ids[ix], all_grades[ix], cluster, band)
        rows.extend(metrics)
        candidates.extend(shortlist)
        if position % 100 == 0 or position == len(eligible):
            print(f"{position}/{len(eligible)} clústeres evaluados", flush=True)

fold_metrics = pd.DataFrame(rows)
if len(fold_metrics) != len(eligible) * len(SEEDS) * 2 * len(TRAIN_SIZES) * len(METHODS):
    raise RuntimeError("No se evaluaron todas las particiones.")

## 4. Comparar y exportar

En `summary.csv`, *recall* es la fracción de referencias recuperada y *lift* el cociente respecto a la selección aleatoria. Los mínimos y máximos muestran cuánto cambia el *recall* entre las tres repeticiones. `unknown_for_review.csv` contiene únicamente los Unknown que entraron en el presupuesto en la primera repetición natural. Su score es local a método y mitad: no usarlo para ordenar clústeres entre sí.

In [ ]:
totals = ["n_eval", "n_review", "ref_ab", "ref_abc", "chance_ab", "chance_abc",
          "hit_a", "hit_b", "hit_c", "hit_ab", "hit_abc"]
by_seed = fold_metrics.groupby(["cohort", "protocol", "method", "seed"], sort=True)[totals].sum().reset_index()
for label in ("ab", "abc"):
    by_seed[f"recall_{label}"] = by_seed[f"hit_{label}"] / by_seed[f"ref_{label}"].replace(0, np.nan)
    by_seed[f"random_recall_{label}"] = by_seed[f"chance_{label}"] / by_seed[f"ref_{label}"].replace(0, np.nan)
    by_seed[f"lift_{label}"] = by_seed[f"hit_{label}"] / by_seed[f"chance_{label}"].replace(0, np.nan)

summary = by_seed.groupby(["cohort", "protocol", "method"], sort=True).agg(
    n_eval=("n_eval", "first"), n_review=("n_review", "first"),
    ref_ab=("ref_ab", "first"), ref_abc=("ref_abc", "first"),
    hits_ab_mean=("hit_ab", "mean"), random_hits_ab_mean=("chance_ab", "mean"),
    hits_abc_mean=("hit_abc", "mean"), random_hits_abc_mean=("chance_abc", "mean"),
    recall_ab=("recall_ab", "mean"), recall_ab_min=("recall_ab", "min"), recall_ab_max=("recall_ab", "max"),
    recall_abc=("recall_abc", "mean"), recall_abc_min=("recall_abc", "min"), recall_abc_max=("recall_abc", "max"),
    lift_ab=("lift_ab", "mean"), lift_abc=("lift_abc", "mean"),
).reset_index()
ranked_unknown = pd.DataFrame(candidates, columns=[
    "cohort", "cluster", "fold", "method", "rank_in_fold", "object_id", "score",
])
OUTPUT.mkdir(parents=True, exist_ok=True)
for name, table in {
    "cluster_inventory": inventory, "fold_metrics": fold_metrics,
    "by_seed": by_seed, "summary": summary, "unknown_for_review": ranked_unknown,
}.items():
    table.to_csv(OUTPUT / f"{name}.csv", index=False)

run = {
    "birch_manifest": manifest, "input_sha256": hashes,
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "seeds": SEEDS, "budget_fraction": BUDGET, "training_sizes": TRAIN_SIZES,
    "small_range": [SMALL_MIN, SMALL_MAX], "large_min": LARGE_MIN, "case_cluster": CASE_CLUSTER,
    "isolation_forest_trees": IF_TREES, "lof_neighbors": LOF_NEIGHBORS,
}
(OUTPUT / "run.json").write_text(json.dumps(run, indent=2) + "\n", encoding="utf-8")
print(summary[["cohort", "protocol", "method", "ref_ab", "recall_ab", "lift_ab",
               "ref_abc", "recall_abc", "lift_abc"]].round(3).to_string(index=False))
print(f"Resultados guardados en {OUTPUT}")

## 5. Figura opcional

Las barras de error indican el mínimo y el máximo entre repeticiones; no son intervalos de confianza.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter
except ImportError:
    print("Matplotlib no está disponible; los CSV ya se han generado.")
else:
    bands = list(COHORTS[:2])
    positions = np.arange(len(bands))
    bins = ("Pequeños (50–149)", "Grandes (≥500)")
    bar_width = 0.18
    palette = ("#346C91", "#BE5A2A", "#13836F")
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.55), sharey=True)
    for ax, (protocol, title) in zip(axes, (
        ("natural", "Mitad completa para ajustar"),
        ("matched25", "25 objetos por ajuste"),
    )):
        rows_for_plot = summary.loc[summary.protocol.eq(protocol)]
        for offset, (method, color) in enumerate(zip(METHODS, palette)):
            points = rows_for_plot.loc[rows_for_plot.method.eq(method)].set_index("cohort").loc[bands]
            mean = points["recall_ab"].to_numpy()
            errors = np.vstack((mean - points["recall_ab_min"], points["recall_ab_max"] - mean)).clip(min=0)
            ax.bar(positions + (offset - 1.5) * bar_width, mean,
                   bar_width, label=method, color=color, yerr=errors, capsize=3)
        ax.bar(positions + 1.5 * bar_width, points["random_hits_ab_mean"] / points["ref_ab"],
               bar_width, label="Azar esperado", color="#9CA3AB")
        ax.set_xticks(positions, [f"{name}\n{n} referencias A+B" for name, n in zip(bins, points["ref_ab"])])
        ax.set_title(title)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
        ax.set_axisbelow(True)
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["right", "top"]].set_visible(False)
    axes[0].set_ylabel("Fracción de referencias A+B recuperadas")
    handles, names = axes[0].get_legend_handles_labels()
    fig.legend(handles, names, loc="lower center", ncols=4, frameon=False)
    fig.suptitle("Anomalías en Euclid Q1 | Presupuesto: 10 % por mitad")
    fig.tight_layout(rect=(0, 0.085, 1, 0.92))
    fig.savefig(OUTPUT / "recuperacion_ab.png", dpi=160)
    plt.show()